### New algorithm for enumerating every $\Z_2^n$-colorable PL~spheres of Picard number $5$

In [1]:
using Oscar
using ProgressBars
using SparseArrays


  ___   ___   ___    _    ____
 / _ \ / __\ / __\  / \  |  _ \  | Combining and extending ANTIC, GAP,
| |_| |\__ \| |__  / ^ \ |  ´ /  | Polymake and Singular
 \___/ \___/ \___//_/ \_\|_|\_\  | Type "?Oscar" for more information
o--------o-----o-----o--------o  | Documentation: https://docs.oscar-system.org
  S Y M B O L I C   T O O L S    | Version 1.4.1


In [2]:
function boundary_incidence_facets_to_ridges(facets::Vector{Vector{Int}})
    # normalize facets (sorted, unique)
    F = [sort(unique(f)) for f in facets]
    d = length(F[1]) - 1  # facet dimension
    @assert all(length(f) == d+1 for f in F) "Need a pure complex (all facets same size)."

    # collect ridges (each facet contributes its (d-1)-subfaces by deleting one vertex)
    ridge_dict = Dict{Vector{Int}, Int}()  # ridge -> row index
    ridges = Vector{Vector{Int}}()
    for f in F
        for k in eachindex(f)
            r = [f[i] for i in eachindex(f) if i != k]  # already sorted since f is sorted
            if !haskey(ridge_dict, r)
                push!(ridges, r)
                ridge_dict[r] = length(ridges)
            end
        end
    end

    m = length(ridges)
    n = length(F)

    # build sparse 0/1 matrix (m x n): ridges x facets
    I = Int[]   # row indices
    J = Int[]   # col indices
    V = Int8[]  # values (all ones)

    for (j, f) in pairs(F)
        for k in eachindex(f)
            r = [f[i] for i in eachindex(f) if i != k]
            i = ridge_dict[r]
            push!(I, i)
            push!(J, j)
            push!(V, Int8(1))
        end
    end

    A = sparse(I, J, V, m, n)  # SparseMatrixCSC{Int8}

    return ridges, A
end



boundary_incidence_facets_to_ridges (generic function with 1 method)

In [3]:
function kernel_basis_mod2(A)
    m, n = size(A)

    # Build R = A mod 2 as a BitMatrix
    R = falses(m, n)
    @inbounds for j in 1:n
        for p in nzrange(A, j)
            i = rowvals(A)[p]
            if isodd(A.nzval[p])
                R[i, j] ⊻= true
            end
        end
    end

    # Reduced row echelon form over GF(2), recording pivot columns and pivot rows
    pivcol = Int[]
    pivrow = Int[]
    row = 1
    @inbounds for col in 1:n
        row > m && break

        # find pivot in this column at/under current row
        piv = 0
        for r in row:m
            if R[r, col]
                piv = r
                break
            end
        end
        piv == 0 && continue

        # swap into position
        if piv != row
            @views begin
                tmp = copy(R[row, :])
                R[row, :] .= R[piv, :]
                R[piv, :] .= tmp
            end
        end

        push!(pivcol, col)
        push!(pivrow, row)

        # eliminate this column in all other rows (RREF)
        @views for r in 1:m
            if r != row && R[r, col]
                R[r, :] .= R[r, :] .!= R[row, :]   # XOR over GF(2)
            end
        end

        row += 1
    end

    # free columns = variables you set to 1 to generate kernel basis vectors
    pivot_set = Set(pivcol)
    free_cols = [j for j in 1:n if !(j in pivot_set)]

    basis = BitVector[]
    isempty(free_cols) && return basis  # full column rank over GF(2)

    # Build one kernel vector per free column
    @inbounds for f in free_cols
        x = falses(n)
        x[f] = true

        # back-substitute using RREF rows: x[pivot] = sum(other entries in row) mod 2
        for t in length(pivcol):-1:1
            c = pivcol[t]
            r = pivrow[t]
            @views begin
                # parity of dot(R[r, :], x) excluding pivot c
                tmp = R[r, :] .& x
                tmp[c] = false
                x[c] = isodd(count(tmp))
            end
        end

        push!(basis, x)
    end

    return basis
end


kernel_basis_mod2 (generic function with 1 method)

In [4]:
using SparseArrays

"""
    kernel_elements_with_Ax_in_0_or_2_noprune(A, basis) -> Vector{BitVector}

Full enumeration WITHOUT pruning.

- A is strictly 0/1 and used over ℤ for Ax.
- basis is a GF(2)-basis of ker(A mod 2), given as Vector{BitVector}.
- Enumerates all nonzero x in the GF(2)-span of `basis` and returns those with Ax ∈ {0,2}^m.
- Complexity: Θ(2^k * (cost to update/check)), where k = length(basis).
"""
function kernel_elements_with_Ax_in_0_or_2(
    A::SparseMatrixCSC{<:Integer},
    basis::Vector{BitVector},
)
    m, n = size(A)
    k = length(basis)
    println("Dimension of the kernel: ",k)
    @assert all(length(b) == n for b in basis) "Basis vectors must have length = #cols(A)."

    # column -> incident rows (A is 0/1)
    rv = rowvals(A)
    colrows = Vector{Vector{Int}}(undef, n)
    for j in 1:n
        colrows[j] = [rv[p] for p in nzrange(A, j)]
    end

    # supports of basis vectors (columns they toggle)
    supp = Vector{Vector{Int}}(undef, k)
    for t in 1:k
        supp[t] = findall(basis[t])
    end

    x = falses(n)
    counts = zeros(Int16, m)  # Ax over ℤ
    sols = BitVector[]

    function toggle_basis!(t::Int)
        @inbounds for j in supp[t]
            turning_on = !x[j]
            x[j] = turning_on
            δ = turning_on ? Int16(1) : Int16(-1)
            for i in colrows[j]
                counts[i] += δ
            end
        end
    end

    @inline function feasible_final()::Bool
        @inbounds for i in 1:m
            c = counts[i]
            if !(c == 0 || c == 2)
                return false
            end
        end
        return true
    end

    function dfs(t::Int, any_one::Bool)
        if t > k
            if any_one && feasible_final()
                push!(sols, copy(x))
            end
            return
        end

        # branch 0: do not include basis[t]
        dfs(t + 1, any_one)

        # branch 1: include basis[t]
        toggle_basis!(t)
        dfs(t + 1, true)
        toggle_basis!(t)  # toggle back (self-inverse over GF(2))
    end

    dfs(1, false)  # excludes zero vector by any_one=false
    return sols
end


kernel_elements_with_Ax_in_0_or_2

In [5]:
# If homology is UNREDUCED in your setup (most common):
function is_homology_sphere(K::Oscar.SimplicialComplex)
    b = betti_numbers(K)
    d = dim(K)
    for i in 0:d
        if i == 0 || i == d
            b[i+1] == 1 || return false
        else
            b[i+1] == 0 || return false
        end
    end
    return true
end


is_homology_sphere (generic function with 1 method)

In [ ]:
using SparseArrays

"""
    enumerate_kernel_Ax_0or2_with_mandatory_facets(A, basis; mandatory_facets, forbid_implied=true)

Enumerate all x in span_GF(2)(basis) ⊆ {0,1}^n such that:
1) x[j]=1 for all j in `mandatory_facets`,
2) Ax over ℤ has every entry in {0,2}.

Speedups:
- Deterministic propagation from the 0/2 row-sum constraint to infer additional forced
  mandatory/forbidden facets (sound preprocessing).
- Linear reduction: restrict the GF(2) kernel span to an affine subspace satisfying the
  (mandatory + forbidden) facet constraints via GF(2) elimination on coefficient space.

Inputs:
- A: m×n sparse 0/1 ridge×facet incidence matrix (SparseMatrixCSC)
- basis: Vector{BitVector} (length n each), a GF(2)-basis of ker(A mod 2)
- mandatory_facets: Vector{Int} (facet indices forced to 1)
- forbid_implied: if true, uses propagation to infer forbidden facets too

Output:
- Vector{BitVector} of all solutions x (each length n)

Notes:
- No pruning in the final enumeration step (full enumeration of the reduced affine space).
- Throws error if the facet constraints are inconsistent with the kernel span.
"""
function enumerate_kernel_Ax_0or2_with_mandatory_facets(
    A::SparseMatrixCSC{<:Integer},
    basis::Vector{BitVector};
    mandatory_facets::Vector{Int},
    forbid_implied::Bool = true,
)
    m, n = size(A)
    k = length(basis)
    @assert all(length(b) == n for b in basis) "Each basis vector must have length n=#cols(A)."

    # ---------- adjacency: col->rows and row->cols ----------
    rv = rowvals(A)
    colrows = Vector{Vector{Int}}(undef, n)
    for j in 1:n
        colrows[j] = [rv[p] for p in nzrange(A, j)]
    end
    rowcols = [Int[] for _ in 1:m]
    for j in 1:n
        for i in colrows[j]
            push!(rowcols[i], j)
        end
    end

    # ---------- preprocessing propagation (0/2 row sums) ----------
    # state: -1 forbidden, 0 unknown, +1 mandatory
    state = fill(Int8(0), n)
    for j in mandatory_facets
        state[j] = 1
    end

    function propagate_0or2!()
        # queue rows affected by assignments: initialize with rows touched by initial mandatory facets
        inqueue = fill(false, m)
        queue = Int[]
        for j in mandatory_facets
            for i in colrows[j]
                if !inqueue[i]
                    push!(queue, i); inqueue[i] = true
                end
            end
        end
        # If no mandatory facets, still do nothing (no forced info).
        isempty(queue) && return true

        while !isempty(queue)
            i = pop!(queue)
            inqueue[i] = false

            cols = rowcols[i]
            s = 0
            u = 0
            last_unknown = 0
            @inbounds for j in cols
                if state[j] == 1
                    s += 1
                elseif state[j] == 0
                    u += 1
                    last_unknown = j
                end
            end

            if s > 2
                return false
            elseif s == 2
                # force all other incident facets to 0
                @inbounds for j in cols
                    if state[j] == 0
                        state[j] = -1
                        # enqueue rows affected by setting facet j
                        for r in colrows[j]
                            if !inqueue[r]
                                push!(queue, r); inqueue[r] = true
                            end
                        end
                    elseif state[j] == 1
                        # ok
                    end
                end
            elseif s == 1
                if u == 0
                    return false
                elseif u == 1
                    # forced 1
                    j = last_unknown
                    if state[j] == -1
                        return false
                    end
                    state[j] = 1
                    for r in colrows[j]
                        if !inqueue[r]
                            push!(queue, r); inqueue[r] = true
                        end
                    end
                end
            else # s == 0
                if u == 1
                    # cannot achieve 2 with a single unknown -> forced 0
                    j = last_unknown
                    if state[j] == 1
                        return false
                    end
                    state[j] = -1
                    for r in colrows[j]
                        if !inqueue[r]
                            push!(queue, r); inqueue[r] = true
                        end
                    end
                end
            end
        end

        return true
    end

    if forbid_implied
        ok = propagate_0or2!()
        ok || return BitVector[]
    end

    mand = findall(==(Int8(1)), state)
    forb = findall(==(Int8(-1)), state)

    # ---------- GF(2) affine restriction on coefficient space y ----------
    # x = sum_t y[t]*basis[t] (mod2). Constraints: x[j]=rhs_j for j in mand∪forb.
    C = vcat(mand, forb)
    rhs = BitVector(vcat(fill(true, length(mand)), fill(false, length(forb))))
    r = length(C)

    # Build constraint matrix Mmat[r,k] with Mmat[ri,t] = basis[t][C[ri]]
    Mmat = falses(r, k)
    for (ri, j) in enumerate(C)
        @inbounds for t in 1:k
            Mmat[ri, t] = basis[t][j]
        end
    end

    # Solve Mmat*y = rhs over GF(2): RREF
    function solve_affine_gf2(M::BitMatrix, b::BitVector)
        rr, kk = size(M)
        A2 = copy(M)
        b2 = copy(b)

        pivot_rows = Int[]
        pivot_cols = Int[]

        row = 1
        for col in 1:kk
            row > rr && break
            piv = 0
            for rrr in row:rr
                if A2[rrr, col]
                    piv = rrr; break
                end
            end
            piv == 0 && continue

            if piv != row
                @views begin
                    tmp = copy(A2[row, :]); A2[row, :] .= A2[piv, :]; A2[piv, :] .= tmp
                end
                b2[row], b2[piv] = b2[piv], b2[row]
            end

            push!(pivot_rows, row)
            push!(pivot_cols, col)
end
            @views for rrr in 1:rr
                if rrr != row && A2[rrr, col]
                    A2[rrr, :] .= A2[rrr, :] .!= A2[row, :]
                    b2[rrr] = xor(b2[rrr], b2[row])
                end
            end

            row += 1
        end

        for rrr in 1:rr
            if !any(@view A2[rrr, :]) && b2[rrr]
                error("Inconsistent facet constraints inside kernel span (no solution).")
            end
        end

        pivot_set = Set(pivot_cols)
        free_cols = [c for c in 1:kk if !(c in pivot_set)]

        y0 = falses(kk)
        for (pr, pc) in zip(pivot_rows, pivot_cols)
            y0[pc] = b2[pr]
        end

        yfree = BitVector[]
        for f in free_cols
            y = falses(kk)
            y[f] = true
            for (pr, pc) in zip(pivot_rows, pivot_cols)
                y[pc] = A2[pr, f]
            end
            push!(yfree, y)
        end

        return y0, yfree
    end

    y0, yfree = solve_affine_gf2(Mmat, rhs)

    # combine basis vectors according to coefficient vector y
    function combine_x(y::BitVector)
        x = falses(n)
        @inbounds for t in 1:k
            if y[t]
                x .= x .!= basis[t]  # XOR for Bool vectors
            end
        end
        return x
    end

    x0 = combine_x(y0)
    gens = [combine_x(u) for u in yfree]  # reduced generators in x-space

    # ---------- final enumeration in the reduced affine space, checking Ax in {0,2} ----------
    # initialize counts = A*x0 over ℤ
    counts = zeros(Int16, m)
    for j in findall(x0)
        @inbounds for i in colrows[j]
            counts[i] += Int16(1)
        end
    end
    # early reject
    for i in 1:m
        counts[i] > 2 && return BitVector[]
    end

    # generator supports (columns toggled)
    supp = [findall(g) for g in gens]
    tgens = length(gens)

    x = copy(x0)
    sols = BitVector[]

    function toggle_gen!(t)
        @inbounds for j in supp[t]
            turning_on = !x[j]
            x[j] = turning_on
            δ = turning_on ? Int16(1) : Int16(-1)
            for i in colrows[j]
                counts[i] += δ
            end
        end
    end

    function feasible_final()
        @inbounds for i in 1:m
            c = counts[i]
            if !(c == 0 || c == 2)
                return false
            end
        end
        return true
    end

    function dfs(t::Int)
        if t > tgens
            if feasible_final()
                push!(sols, copy(x))
            end
            return
        end
        dfs(t + 1)
        toggle_gen!(t)
        dfs(t + 1)
        toggle_gen!(t)
    end

    dfs(1)
    return sols
end


enumerate_kernel_Ax_0or2_with_mandatory_facets

In [7]:
function enumerate_polygons(facets_list)
    ridges, A = boundary_incidence_facets_to_ridges(facets_list)
    basis = kernel_basis_mod2(A)
    mandatory=[1]
    res = [facets_list[findall(facets_vec)] for facets_vec in enumerate_kernel_Ax_0or2_with_mandatory_facets(A,basis;mandatory_facets=[1]) if is_sphere(simplicial_complex(facets_list[findall(facets_vec)]))]
    println(res)
end

function find_from_lower(facets_list)
    xs = enumerate_kernel_Ax_0or2_with_mandatory_facets(A, basis; mandatory_facets=mandatory)
end

function load_bin_matroids_bases(m)
    path="./binary_matroids/bin_mat_better_" * string(m) * "_" * string(m-5)
    open(path) do io
        [Meta.parse(strip(line)) |> eval
         for line in eachline(io)
         if !isempty(strip(line))]
    end
end


set_matroids=load_bin_matroids_bases(7)

for facets_list in set_matroids
    enumerate_polygons(facets_list)
end


[[[27, 30], [27, 29], [23, 30], [23, 29]], [[27, 30], [25, 27], [12, 30], [12, 25]], [[27, 30], [26, 27], [12, 30], [12, 26]], [[27, 30], [27, 29], [12, 30], [12, 29]], [[27, 30], [25, 27], [23, 30], [23, 29], [12, 29], [12, 25]], [[27, 30], [26, 27], [23, 30], [23, 29], [12, 29], [12, 26]], [[27, 30], [25, 27], [23, 30], [23, 25]], [[27, 30], [27, 29], [23, 29], [23, 25], [12, 30], [12, 25]], [[27, 30], [26, 27], [23, 30], [23, 25], [12, 26], [12, 25]], [[27, 30], [25, 27], [23, 29], [23, 25], [12, 30], [12, 29]], [[27, 30], [27, 29], [23, 30], [23, 25], [12, 29], [12, 25]], [[27, 30], [26, 27], [23, 30], [23, 26]], [[27, 30], [27, 29], [23, 29], [23, 26], [12, 30], [12, 26]], [[27, 30], [25, 27], [23, 30], [23, 26], [12, 26], [12, 25]], [[27, 30], [26, 27], [23, 29], [23, 26], [12, 30], [12, 29]], [[27, 30], [27, 29], [23, 30], [23, 26], [12, 29], [12, 26]], [[27, 30], [26, 27], [23, 26], [23, 25], [12, 30], [12, 25]], [[27, 30], [25, 27], [23, 26], [23, 25], [12, 30], [12, 26]]]
[[[

In [50]:
function load_bin_matroids_bases(m)
    path = "binary_matroids/bin_mat_better_" * string(m) * "_" * string(m-5)
    open(path) do io
        [Meta.parse(strip(line)) |> eval
         for line in eachline(io)
         if !isempty(strip(line))]
    end
end

mat_DB = Dict{Int, Vector{Vector{Vector{Int}}}}()

for m in 7:15   # e.g. ms = 6:31
    mat_DB[m] = load_bin_matroids_bases(m)
end



In [56]:
using Serialization

# save
open("mat_DB.jls", "w") do io
    serialize(io, mat_DB)
end

# load
global mat_DB = open("mat_DB.jls", "r") do io
    deserialize(io)
end

Dict{Int64, Vector{Vector{Vector{Int64}}}} with 9 entries:
  13 => [[[15, 21, 23, 24, 27, 28, 29, 30], [15, 18, 23, 24, 27, 28, 29, 30], […
  15 => [[[19, 20, 23, 24, 26, 27, 28, 29, 30, 31], [17, 20, 23, 24, 26, 27, 28…
  7  => [[[27, 30], [27, 29], [26, 27], [25, 27], [23, 30], [23, 29], [23, 26],…
  11 => [[[12, 26, 27, 28, 29, 30], [12, 25, 27, 28, 29, 30], [12, 25, 26, 27, …
  10 => [[[24, 25, 27, 29, 31], [22, 24, 25, 29, 31], [22, 24, 25, 27, 31], [21…
  9  => [[[20, 24, 29, 31], [20, 22, 29, 31], [20, 22, 24, 29], [19, 24, 29, 31…
  12 => [[[12, 20, 22, 24, 29, 30, 31], [12, 20, 22, 23, 24, 29, 31], [12, 19, …
  8  => [[[20, 24, 31], [19, 24, 31], [19, 20, 31], [19, 20, 24], [12, 20, 31],…
  14 => [[[15, 20, 21, 23, 24, 27, 28, 29, 30], [15, 18, 21, 23, 24, 27, 28, 29…

In [116]:
using PyCall          # optional: used to call polymake if available
using Polymake
const topaz = Polymake.topaz
using ProgressMeter
using Nemo
F = GF(2)

# Construct a matrix from a binary matroid

function binary_matrix_representation(M; B=nothing)
    is_binary(M) || error("Matroid is not binary")

    # ground set
    E = collect(matroid_groundset(M))
    n = length(E)

    # choose basis
    if B === nothing
        B = first(bases(M))
    else
        length(B) == rank(M) || error("Provided set is not a basis")
    end

    r = length(B)

    # index maps
    row_of = Dict(b => i for (i,b) in enumerate(B))
    col_of = Dict(e => j for (j,e) in enumerate(E))

    # initialize matrix
    A = matrix(F,zeros(Int, (r, n)))

    # basis columns = identity
    for b in B
        A[row_of[b], col_of[b]] = F(1)
    end

    # non-basis columns via fundamental circuits
    for e in E
        e in B && continue
        C = fundamental_circuit(M, B, e)
        for b in C
            b in B || continue
            A[row_of[b], col_of[e]] = F(1)
        end
    end

    return A, B, E
end

# ---------------------------
# using Topaz for isom
# ---------------------------

function bases_to_topaz_complex(bases::Vector{Vector{Int}})
    # collect and sort vertices
    verts = sort!(unique(vcat(bases...)))

    # map vertex labels to 0-based indices
    vmap = Dict(v => i-1 for (i,v) in enumerate(verts))

    facets = [ [vmap[x] for x in B] for B in bases ]
    return topaz.SimplicialComplex(FACETS = facets), verts
end

function topaz_isomorphism(basesA, basesB)
    SC_A, vertsA = bases_to_topaz_complex(basesA)
    SC_B, vertsB = bases_to_topaz_complex(basesB)
    length(vertsA) == length(vertsB) || return nothing
    T = topaz.find_facet_vertex_permutations(SC_A, SC_B)
    if T===nothing
        return nothing
    end
    _,p=T
    perm = Dict(
        vertsA[i] => vertsB[p[i]+1]
        for i in eachindex(vertsA)
    )
    return perm
end


# Build canonical vertex list present in a matroid (sorted)
function vertex_list_of_bases(bases::Vector{Vector{Int}})
    s = Set{Int}()
    for b in bases
        for v in b
            push!(s, v)
        end
    end
    return sort(collect(s))
end



# ---------------------------
# Main builder
# ---------------------------
"""
build_iso_db!(Iso_DB, mat_DB; ms = sort(collect(keys(mat_DB))), verbose=false)

- Iso_DB will be mutated (create empty Dict before passing).
- mat_DB: Dict{Int, Vector{Vector{Vector{Int}}}} mapping m -> list of matroid-bases
- For each m in ms (skips if m-1 not present), and each k in mat_DB[m],
  finds first vertex v in 1:m such that corank(contract_at_vertex(M,v)) >= corank(M),
  computes contraction, then finds l and a permutation mapping contraction -> some mat_DB[m-1][l].
- Stores Iso_DB[m][k] = (l, mapping::Dict{Int,Int}) or (-1, nothing) if not found.
"""
function build_iso_db!(Iso_DB::Dict{Int,Dict{Int,Tuple{Int,Int,Any}}}, mat_DB::Dict{Int,Vector{Vector{Vector{Int}}}}; ms=nothing, verbose=false)
    # ms === nothing && (ms = sort(collect(keys(mat_DB))))
    for m in ms
        if !haskey(mat_DB, m-1)
            verbose && @info("Skipping m=$m because mat_DB[$(m-1)] not present")
            continue
        end
        Iso_DB[m] = Dict{Int,Tuple{Int,Any}}()
        @showprogress desc="Computing m=$(m)..." for (k, Mbases) in enumerate(mat_DB[m])
            V=vertex_list_of_bases(Mbases)
            M = matroid_from_bases(Mbases,V)
            found_v = nothing
            Mv = nothing
            # pick smallest v in 1:m satisfying corank(contract) >= corank(M)
            vcontract=nothing
            for v in V
                Mv = contraction(M,v)
                if is_loopless(Mv)
                    found_v=true
                    vcontract=v
                    break
                end
            end
            if found_v === nothing
                vcontract = V[1]
            end
            Mv = contraction(M,vcontract)
            contracted_bases = bases(Mv)

            # search through mat_DB[m-1] for an isomorphism
            perm = nothing
            found_index = -1
            found_isom = false
            for (l, target_bases) in enumerate(mat_DB[m-1])
                M2 = matroid_from_bases(target_bases,vertex_list_of_bases(target_bases))
                if !is_isomorphic(Mv,M2)
                    continue
                end
                perm = topaz_isomorphism(collect(circuits(Mv)),collect(circuits(M2)))
                if perm !== nothing
                    found_index = l
                    break
                end
            end

            if perm === nothing
                Iso_DB[m][k] = (-1, vcontract, nothing)
                A,_,_ = binary_matrix_representation(M)
                display(A)
                verbose && @warn(
                    "No isomorphic matroid found for contraction  m=$m, k=$k at v=$vcontract"
                )
            else
                Iso_DB[m][k] = (found_index,vcontract, perm)
                # verbose && @info(
                #     "Found iso: m=$m, k=$k -> m-1 index=$found_index, v=$vcontract"
                # )
            end
        end
    end
    return Iso_DB
end


build_iso_db!

In [117]:
for list_bases in mat_DB[7]
    A,_,_ = binary_matrix_representation(matroid_from_bases(list_bases, vertex_list_of_bases(list_bases)))
    display(A)
end

[1   1   0   0   1   0   0]
[0   0   1   1   0   1   1]

[0   1   1   1   1   0]
[1   0   1   1   0   1]

[0   1   1   0   1   0]
[1   0   0   1   0   1]

[0   1   1   1   1   1   0]
[1   1   1   1   1   0   1]

[0   0   1   1   1   1   0]
[1   1   1   0   0   0   1]

[1   1   0   1   1   1   0]
[1   1   1   0   1   0   1]

[1   0   1   1   0]
[1   1   0   0   1]

[1   1   0   1   0   0]
[1   1   1   0   1   1]

In [115]:
iso_DB = Dict{Int,Dict{Int,Tuple{Int,Any}}}()

build_iso_db!(iso_DB,mat_DB,ms=8:11,verbose=true)

for m=8:11
    for data in iso_DB[m]
        k,perm = data
        println(perm)
    end
end

Computing m=8...  13%|████▋                              |  ETA: 0:00:01

[1   1   1   0   0   1   0   0]
[1   1   0   0   1   0   1   0]
[1   1   0   1   0   0   0   1]

┌ Warning: No isomorphic matroid found for contraction  m=8, k=6 at v=18
└ @ Main In[112]:158
Computing m=8... 100%|███████████████████████████████████| Time: 0:00:00
Computing m=9... 100%|███████████████████████████████████| Time: 0:00:00
Computing m=10... 100%|██████████████████████████████████| Time: 0:00:04
Computing m=11... 100%|██████████████████████████████████| Time: 0:00:32


(6, Dict(22 => 23, 29 => 28, 25 => 26, 27 => 25, 31 => 30, 12 => 12, 24 => 29))
(2, Dict(22 => 27, 29 => 30, 20 => 26, 31 => 28, 24 => 29, 19 => 25))
(5, Dict(29 => 29, 27 => 27, 26 => 26, 30 => 30, 28 => 28, 23 => 23, 19 => 19))
(7, Dict(20 => 20, 31 => 31, 12 => 12, 24 => 24, 19 => 19))
(-1, nothing)
(2, Dict(6 => 25, 15 => 27, 20 => 28, 29 => 30, 12 => 26, 23 => 29))
(5, Dict(22 => 19, 29 => 30, 25 => 28, 27 => 26, 30 => 29, 28 => 27, 23 => 23))
(5, Dict(22 => 23, 29 => 30, 21 => 19, 25 => 27, 27 => 29, 31 => 26, 24 => 28))
(1, Dict(11 => 26, 9 => 12, 12 => 23, 24 => 29, 30 => 27, 31 => 30, 23 => 25))
(6, Dict(29 => 26, 27 => 25, 26 => 28, 12 => 12, 30 => 29, 28 => 30, 19 => 23))
(4, Dict(22 => 22, 6 => 6, 29 => 29, 25 => 25, 27 => 27, 12 => 12, 24 => 24))
(8, Dict(22 => 23, 29 => 26, 7 => 12, 20 => 27, 31 => 28, 24 => 29))
(2, Dict(29 => 28, 20 => 25, 27 => 30, 24 => 29, 30 => 26, 23 => 27))
(6, Dict(29 => 29, 25 => 25, 26 => 26, 12 => 12, 30 => 30, 28 => 28, 23 => 23))
(6, Dict(29

In [ ]:
struct LevelDB
    mats_bases::Vector{Vector{Vector{Int}}}     # reps as lists of bases
    mats_M::Vector{Matroid}                     # Matroid objects
    Ks::Vector{Vector{Vector{Int}}}             # Ks[i] = list of K as base-indices into mats_bases[i]
end

_base_key(B::Vector{Int}) = join(sort(B), ",")

function base_index_dict(bases::Vector{Vector{Int}})
    d = Dict{String,Int}()
    for (i,B) in enumerate(bases)
        d[_base_key(B)] = i
    end
    return d
end

# choose one v only (smallest non-loop, change if you want)
function choose_v_for_contraction(M::Matroid)
    for v in sort(collect(matroid_groundset(M)))
        if !(v in loops(M))
            return v
        end
    end
    return 0
end

function build_finalDB_single_v(mmin::Int=2, mmax::Int)
    finalDB = Dict{Int,LevelDB}()

    for m in mmin:mmax
        # construct Matroid objects (adjust if your constructor differs)
        mats_M = load_bin_matroids_bases(m) ccds
        Ks = [Vector{Vector{Int}}() for _ in 1:length(mats_bases)]
        finalDB[m] = LevelDB(mats_bases, mats_M, Ks)

        # base level: all K with Ax in {0,2} and no mandatory facets
        if m == 2
            for i in eachindex(mats_bases)
                basesM = mats_bases[i]
                _, A = boundary_incidence_facets_to_ridges(basesM)
                kerbasis = kernel_basis_mod2(A)
                xs = enumerate_polygons(A, kerbasis)
                finalDB[m].Ks[i] = [findall(x) for x in xs]
            end
            continue
        end

        prev = finalDB[m-1]

        for i in eachindex(mats_bases)
            basesM = mats_bases[i]
            M      = mats_M[i]

            _, A = boundary_incidence_facets_to_ridges(basesM)
            kerbasis = kernel_basis_mod2(A)
            b2iM = base_index_dict(basesM)

            v = choose_v_for_contraction(M)
            v == 0 && continue

            Mv = contraction(M, v)

            # find a rep R at level m-1 isomorphic to Mv, and get an isomorphism phi: R -> Mv
            jprev = 0
            phi = nothing
            for (j, R) in enumerate(prev.mats_M)
                if is_isomorphic(R, Mv)
                    jprev = j
                    phi = isomorphism(R, Mv)
                    break
                end
            end
            jprev == 0 && continue

            basesR = prev.mats_bases[jprev]
            out = Vector{Vector{Int}}()
            seen = Set{String}()

            # for each previously found L in R, force L∪{v} in M (after transport)
            for Lidxs in prev.Ks[jprev]
                mand = Int[]
                ok = true
                for bi in Lidxs
                    Brep = basesR[bi]                     # base in R
                    Bmv = sort!([image(phi, e) for e in Brep])  # transported base in Mv
                    Blift = sort!(vcat(Bmv, [v]))          # base in M
                    ii = get(b2iM, _base_key(Blift), 0)
                    if ii == 0
                        ok = false
                        break
                    end
                    push!(mand, ii)
                end
                ok || continue

                xs = enumerate_kernel_Ax_0or2_with_mandatory_facets(A, kerbasis; mandatory_facets=mand)
                for x in xs
                    idxs = findall(x)
                    key = join(idxs, ",")
                    if !(key in seen)
                        push!(seen, key)
                        push!(out, idxs)
                    end
                end
            end

            finalDB[m].Ks[i] = out
        end
    end

    return finalDB
end


build_finalDB_single_v (generic function with 1 method)

In [16]:
build_finalDB_single_v(7,9)


LoadError: MethodError: no method matching isomorphism(::Matroid, ::Matroid)
The function `isomorphism` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  isomorphism([91m::Type{PermGroup}[39m, [91m::PermGroup[39m; on_gens)
[0m[90m   @[39m [35mOscar[39m [90m~/.julia/packages/Oscar/T5Jnd/src/Groups/[39m[90m[4mhomomorphisms.jl:616[24m[39m
[0m  isomorphism([91m::Type{PermGroup}[39m, [91m::WeylGroup[39m; on_gens)
[0m[90m   @[39m [35mOscar[39m [90m~/.julia/packages/Oscar/T5Jnd/experimental/LieAlgebras/src/[39m[90m[4mWeylGroup.jl:255[24m[39m
[0m  isomorphism([91m::Type{FinGenAbGroup}[39m, [91m::FinGenAbGroup[39m; on_gens)
[0m[90m   @[39m [32mHecke[39m [90m~/.julia/packages/Hecke/jmVIY/src/GrpAb/[39m[90m[4mGrpAbFinGen.jl:648[24m[39m
[0m  ...


In [86]:
methods(isomorphism)
names(Oscar) |> x -> filter(s -> occursin("iso", String(s)), x)

131-element Vector{Symbol}:
 :CartierDivisor
 :Divisor
 :Divisors
 :EffectiveCartierDivisor
 :ToricDivisor
 :ToricDivisorClass
 :WeilDivisor
 :ambient_isometry
 :anticanonical_divisor
 :anticanonical_divisor_class
 :basis_isotypical_component
 :canonical_divisor
 :canonical_divisor_class
 ⋮
 :set_is_isomorphic_with_symmetric_group
 :toric_divisor
 :toric_divisor_class
 :torusinvariant_cartier_divisor_group
 :torusinvariant_prime_divisors
 :torusinvariant_weil_divisor_group
 :trace_lattice_with_isometry
 :trace_lattice_with_isometry_and_transfer_data
 :trivial_divisor
 :trivial_divisor_class
 :weil_divisor
 :zero_divisor